In [ ]:
import geopandas as gpd
import pandas as pd
import rasterio
from pathlib import Path
import xarray as xr
import rioxarray

In [ ]:
refpoints_csv = "..//..//WRIJ_RR_Unpaved_methode_01_data//rr_data_scenarios//scenarios//#SCENARIO#//#SCENARIO#_gebiedsindeling_RR_KNOPEN_tbv_Onderrand.csv"
raster_folder = "..//..//WRIJ_RR_Unpaved_methode_01_data//rr_data_scenarios//scenarios//#SCENARIO#//seepage//"
output_csv = "..//..//WRIJ_RR_Unpaved_methode_02_input//rr_input_scenarios//scenarios//#SCENARIO#//kwel_per_RR_knoop.csv"

In [ ]:
crs = "EPSG:28992"

In [ ]:
# selectie_gebied = 0 # Oude IJssel
# selectie_gebied = 1 # West
# selectie_gebied = 2 # Centraal
# selectie_gebied = 3 # Oost

selectie_gebieden = [1, 2, 3]

scenarios = ["REF", "SCEN"]
# scenarios = ["REF"]

start_date = "2012-4-1"
# end_date = "2015-11-1"
end_date = "2013-4-1"
# end_date = "2018-12-1"

# path to the package containing the dummy-data
dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\"

In [ ]:
date_range_data = pd.date_range(start_date, end_date, freq="MS")

teskts

In [ ]:
ds = xr.Dataset()
for scenario in scenarios:
    print(scenario)
    dir_seepage = Path(raster_folder.replace("#SCENARIO#", scenario))
    data_arrays = []
    for date in date_range_data:
        print(date)
        seepage_raster_filename = f"{scenario}_FLUX_L1L2_{date.strftime('%Y%m')}_MMD.ASC"
        da = rioxarray.open_rasterio(Path(dir_seepage, seepage_raster_filename), masked=True).squeeze("band")
        data_arrays.append(da)
    da = xr.concat(data_arrays, dim="time")
    ds[scenario] = da.assign_coords(time=date_range_data)

tekst

In [ ]:
df_results = {}
for scenario in scenarios:
    print(scenario)
    refpoints_path = Path(refpoints_csv.replace("#SCENARIO#", scenario))
    refpoints = pd.read_csv(refpoints_path, sep=";")
    refpoints = refpoints.rename(columns={"xcoor": "x", "ycoor": "y"})

    ds_refpoints = refpoints.set_index(["x", "y"])["ID_RR_KNOOP"].to_xarray()
    ds[f"{scenario}_refpoints"] = ds_refpoints.reindex_like(ds)

    da_result = ds[scenario].where(ds[f"{scenario}_refpoints"].notnull()).groupby(ds[f"{scenario}_refpoints"]).mean(dim=("stacked_x_y"))
    df_result = da_result.to_dataframe()[scenario].reset_index().pivot(index="time", columns=f"{scenario}_refpoints", values=scenario)
    df_result.columns = ["sep_" + col for col in df_result.columns]
    df_results[scenario] = df_result
df_results = pd.concat(df_results, axis=1)

toevoegen 2010-2012

In [ ]:
df_results_mean = df_results.groupby(df_results.index.month).mean()

In [ ]:
df_start = pd.DataFrame(index=pd.date_range("2010-4-1", start_date, freq="MS"), columns=df_results.columns).iloc[:-1]

In [ ]:
df_temp = df_results_mean.loc[df_start.index.month]
df_temp.index = df_start.index
df_results_total = pd.concat([df_temp, df_results])

wegschrijven per scenario

In [ ]:
for scenario in scenarios:
    df_results_total[scenario].to_csv(output_csv.replace("#SCENARIO#", scenario))